# Birthday Problem
 > Created: Fall 2021, Fermilab LPC, Harrison B. Prosper<br>
> Updated: May 2022, INFN School of Statistics, Paestum, Italy<br>
> Updated: Jun 16-18 2026, INFN School of Statistics, Caserta, Italy<br>

## The Classic Birthday Problem
What's the probability that in a randomly assembled crowd of $n$ people at least two of them have the same birthday? How large must the crowd be for that probability to be at least 50%? 

Assumptions:

  1. There are $B$ possible birthdays (usually, $B = 365$).
  2. Birthdays are equally probable.
  3. People are distinguishable.

## A Variation
What's the probability to have *exactly* 2 people with identical birthdays, while the birthdays of all others in the crowd are unique? What crowd size maximizes the chance to have only a single duplicate birthday.

Here we use the Python module __itertools__ to simulate the problem.

## Exercise

Derive expressions for the probabilities $p_0, p_1, p_2$ of no duplicates, exactly one duplicate, and at least one duplicate, respectively, in terms of $B$ birthdays and crowd size $n$ and compare the result of your expression with the probabilities found below through simulation. Generalize to the probability to have exactly $k \le n$ people with the same birthdays and all others unique.

Try to work out the combinatorics before looking at the solutions.

In [1]:
import numpy as np
from collections import Counter

# combinatorial analysis
from itertools import permutations, combinations, product
from math import factorial, comb

## Create all crowds of size $n$ with $B$ possible birthdays

In [2]:
B = 9  # number of birthdays
n = 6  # crowd size

def create_crowds(B, n):
    print('-'*60)
    print(f'number of birthdays: {B:5d}')
    print(f'crowd size:          {n:5d}')
    print('-'*60)
    
    # initialize an n-tuple: b = (z1,...,zn)
    b = range(1, B+1)
    
    # create all possible n-tuples using "b": (b1,...,bn)
    # Cartesian product of n instances of "b"
    p = product(b, repeat=n)
    
    # verify that number of crowds is K**n
    crowds = []
    while 1:
        try:
            crowd = next(p)
        except StopIteration:
            break
        crowds.append(crowd)
            
    # number of crowds; cardinality of Omega
    T = len(crowds)
    print(f'number of possible crowds: {T:,}, {B**n:,}')
    return  crowds

crowds = create_crowds(B, n)

------------------------------------------------------------
number of birthdays:     9
crowd size:              6
------------------------------------------------------------
number of possible crowds: 531,441, 531,441


## Count crowds with exactly $k$ identical birthdays
Suppose that a crowd of size $n$ has exactly $k$ identical birthdays, while all the remaining $n-k$ birthdays are unique. If one counts how often different birthdays occur in the crowd, and call the array of such counts, *counts*, then such crowds will have a counts array with the following structure: $[\cdots, j,1,\cdots,1]$, where $j > 1$ is the number of times some birthday occurs in the list with all the other counts equal to one. Here's an algorithm to identify integer arrays of this form.

  1. Count how often different integers occur in `counts`. 
  2. Remove *first* instance of desired $k$ from `counts`. 
  3. For the crowds of interest, the array `counts` should now contain a sequence of ones only. Therefore, the sum of `counts` should equal its length.

In [3]:
def matches_pattern(lst, k):
    """True iff exactly one value appears k times and all others appear once."""
    # Get counts of each integer within the list "lst"
    counts = list(Counter(lst).values())

    try:
        counts.remove(k)
    except:
        # k does not appear in counts
        return False

    if len(counts) == 0:
        # Everyone has the same birthday!
        return True
        
    return sum(counts) == len(counts)

In [4]:
a1 = [1,1,3,1,0]
a2 = [1,2,3,4,5]
a3 = [1,2,2,1,5]
a4 = [2,2,2,2,2]

for a, j in [(a1,3), (a2,3), (a3,3), (a4, 5)]:
    print(a)
    print(matches_pattern(a, k=j))
    print()

[1, 1, 3, 1, 0]
True

[1, 2, 3, 4, 5]
False

[1, 2, 2, 1, 5]
False

[2, 2, 2, 2, 2]
True



In [5]:
def count_crowds(crowds, K):
    '''
    crowds:       list of crowds (indexed by birthdays within crowd)
    K:            number of persons, k=1,...,K, with identical birthdays 
    '''

    assert(K > 0)
    
    # list to keep track of crowd counts
    N  = [0]*(K+1)

    # crowd size
    n  = len(crowds[0])
    
    for crowd in crowds:

        for i in range(K):

            # count number of crowds with exactly k birthdays the same,
            # while all others are unique
            k = i + 1
            if matches_pattern(crowd, k):
                N[i] += 1

        if len(set(crowd)) < n:
            # count number of crowds with at least one duplicate
            N[K] += 1
            
    T = len(crowds)
    p = [0]*(K+1)
    for i in range(K+1):
        p[i] = N[i] / T

    print(f'cardinality of Omega:\t\t     {T:10,d}')
    print('-'*68)
    for i in range(K):
        s = ': ' if i == 1 else 's:'
        print(f'number of crowds with {i:d} duplicate{s}  {N[i]:10,d}\tprob: {p[i]:6.3f}')
    print(f'number of crowds with >=1 duplicates:{N[K]:10,d}\tprob: {p[k]:6.3f}')
    return N, p

N, p = count_crowds(crowds, K=4)

cardinality of Omega:		        531,441
--------------------------------------------------------------------
number of crowds with 0 duplicates:      60,480	prob:  0.114
number of crowds with 1 duplicate:      226,800	prob:  0.427
number of crowds with 2 duplicates:      60,480	prob:  0.114
number of crowds with 3 duplicates:       7,560	prob:  0.014
number of crowds with >=1 duplicates:   470,961	prob:  0.886


## Classic birthday problem: solution

If $B$ is the number of possible birthdays, the cardinality, $T = |\Omega|$, of $\color{blue}{\Omega}$, which is the set of all possible crowds of size $n$, is
$$T = B^n,$$
since the birthday of a randomly chosen person will be one of $B$ possible birthdays, and this is true for every person in the crowd. 
The cardinality $K$ of the set of *no* duplicates is determined through the following reasoning.  The first person can have one of $B$ birthdays. The second person can have one of $B-1$ birthdays because we must exclude the birthday of the first person. The third person can have one of $B-2$ birthdays because we must exclude the birthdays of the first and second persons, and likewise for the remaining persons. The $n^\text{th}$ person has one of $B - (n-1)$ birthdays. Therefore, 
\begin{align}
    K & = B \, (B-1) \, \cdots (B - (n-1)),\\
        & = \frac{B \, (B-1) \, \cdots (B - n + 1) \, (B - n)!}{(B - n)!},\\
        & = \frac{B!}{(B-n)!}.
\end{align}
Consequently, assuming every crowd is equally probable, the probability to have at least one duplicate birthday is
\begin{align}
    p & = \frac{T - K}{T},\\
        & = 1 - \frac{B!}{(B-n)! B^n}.
\end{align}

In [6]:
def number_crowds_with_zero_duplicates(B, n):
    return factorial(B) // factorial(B-n)    

K = number_crowds_with_zero_duplicates(B, n)
T = B**n   # Cardinality of the set of all crowds

N = T - K
p = N / T
print(f'number of crowds with at least one duplicate: {N:10,d}\tprob: {p:10.3f}')
print()
print('This agrees with the explicit enumeration above.')

number of crowds with at least one duplicate:    470,961	prob:      0.886

This agrees with the explicit enumeration above.


### Crowd size for which $p \ge 0.5$ for $B = 365$ possible birthdays

In [7]:
B = 365
for n in range(1, 51):
    K = number_crowds_with_zero_duplicates(B, n)
    T = B**n
    q = K / T
    p = 1 - q
    if p >= 0.5:
        print(f'crowd size for which p >= 0.5: {n:5d}\tprob: {p:10.3f}')
        break

crowd size for which p >= 0.5:    23	prob:      0.507


## Exactly $k$ people with the same birthday: Solution

Let's work out the cardinality $N$ of the set of crowds with *exactly* $k$ people with the same birthday while all other $n - k$ birthdays are different.  We've already computed the cardinality, $K$, of the set of crowds of size $n$ with *no* duplicates. If the size of the sub-crowds is $n-k$, and remembering that only $B-1$ birthdays are available for the sub-crowds, then 
\begin{align}
    K & = \frac{(B-1)!}{(B - n + k - 1)!}.
\end{align}
But now we come to a bit of reasoning that is easily derailed. (It derailed me at first!) For a given set of unique birthdays, there are many sub-crowds of size $n - k$ because every person is unique. The number of ways, $M$, to construct sub-crowds of size $n-k$ given a crowd size of $n$, or equivalently, to construct sub-crowds of size $k$ wherein the birthdays are identical, is
\begin{align}
    M & = \binom{n}{n-k},\nonumber\\
    & = \binom{n}{k}.
\end{align}
Finally, there are $B$ ways to have $k$ people with the same birthday. 
Therefore, the number of crowds with $k \leq n$ persons with the same birthday is

\begin{align}
    N & = B M K,\nonumber\\
        & = B \binom{n}{k} \frac{(B-1)!}{(B - n + k-1)!},\nonumber\\
        & = \binom{n}{k} \frac{B!}{(B - n + k-1)!} .
\end{align}

Another way to reason about this is as follows:
  * There are $\binom{n}{k}$ ways to construct sub-crowds of size $k$ of persons with the same birthday in crowds of size $n$.
  * For every one of these sub-crowds, there are $B$ possible birthdays. So the total number of sub-crowds accounting for the number of possible birthdays is $B \binom{n}{k}$.
  * For each of these sub-crowds there are $B-1$ birthdays available for the complementary sub-crowd of size $n-k$ of persons with unique birthdays. Therefore, using the result we derived earlier, there are $(B-1)! / (B-1 + n + k)!$ sub-crowds of persons with unique birthdays.
  * Therefore, the number of distinct crowds with $k$ persons with the same birthday is as stated above.
    
Combinatoric reasoning can be tricky!

In [8]:
def combinatorial_count(n, k, B):
    if k == 1:
        if n > B:
            return 0
        return number_crowds_with_zero_duplicates(B, n)
    else:
        if n - k > B - 1:
            return 0
        return comb(n, k) * number_crowds_with_zero_duplicates(B, n + 1 - k)

In [9]:
B = 9  # number of birthdays
n = 6  # crowd size
K = 4  # number of identical birthdays

T = B**n # cardinality of set Omega

crowds = create_crowds(B, n)
print('\n\tExplicit eumeration')
print()
count_crowds(crowds, K)

print('\n\tAlgebraically computed')
print()
for j in range(1, K+1):
    N = combinatorial_count(n, j, B)
    p = N / T
    print(f'number of crowds with {j-1} duplicates:  {N:10,d}\tprob: {p:10.3f}')

------------------------------------------------------------
number of birthdays:     9
crowd size:              6
------------------------------------------------------------
number of possible crowds: 531,441, 531,441

	Explicit eumeration

cardinality of Omega:		        531,441
--------------------------------------------------------------------
number of crowds with 0 duplicates:      60,480	prob:  0.114
number of crowds with 1 duplicate:      226,800	prob:  0.427
number of crowds with 2 duplicates:      60,480	prob:  0.114
number of crowds with 3 duplicates:       7,560	prob:  0.014
number of crowds with >=1 duplicates:   470,961	prob:  0.886

	Algebraically computed

number of crowds with 0 duplicates:      60,480	prob:      0.114
number of crowds with 1 duplicates:     226,800	prob:      0.427
number of crowds with 2 duplicates:      60,480	prob:      0.114
number of crowds with 3 duplicates:       7,560	prob:      0.014
